# SCRES-IA David Playground: Track A, Track B, PPO, DMLPA, and fast experiment loops

This notebook is meant for **David** and for collaborators who want to run quick, editable experiments on the SCRES-IA / Garrido supply-chain resilience simulator.

It is intentionally verbose. The goal is not to hide complexity behind one magic command. The goal is to make every important choice visible:

- What is **Track A**?
- What is **Track B**?
- What are the decision variables?
- What defaults are currently the best-known defaults?
- Which defaults are paper-facing, and which are only exploratory?
- How can David edit the model architecture, including a DMLPA-style Transformer feature extractor?
- How do we run a tiny smoke test in Colab/Kaggle without wasting compute?
- How do we launch a more serious run with the repository runners when the smoke test works?

The notebook has two execution styles:

1. **Notebook-native sandbox**: edit the model in a cell, train for a few timesteps, and verify that training/evaluation works. This is for fast iteration.
2. **Repository runners**: call the audited CLI scripts that produce richer CSV/JSON artifacts. This is for reproducible experiments and later paper evidence.

Important boundary:

- The notebook is **not** a confirmatory paper run by default.
- Defaults are deliberately small so David can verify that code runs, observations/actions make sense, and models learn without waiting hours.
- For claims, use the repository runners with more seeds, dense static frontiers, common random numbers, and Garrido-style metrics.

## Current scientific state, in plain language

### Track A: Garrido original decision family

Track A is the thesis-faithful or thesis-adjacent lane. It uses Garrido-style strategic variables:

- inventory / buffer decisions, usually around the thesis inventory periods `I0`, `I168`, `I336`, `I504`, `I672`, `I1344`;
- manufacturing capacity shifts `S1`, `S2`, `S3`;
- in richer Track A variants, per-operation buffers for `Op3`, `Op5`, and `Op9`.

What we learned so far:

- Track A is very useful scientifically because it tells us where RL **does not** add value.
- Dense static frontiers repeatedly captured the best achievable behavior with Garrido's buffer/shift variables.
- PPO could often learn the same plateau, especially after behavior cloning / warm-start, but it did not produce a robust publishable win over the best dense static frontier.
- Best-known Track A repair result was essentially a technical tie: dynamic around `0.155247` vs best static around `0.155254` in the per-op conflict campaign.
- Therefore Track A is currently a **boundary characterization**: neural learning value is frontier-dependent; if the action space does not reach the binding constraint, RL cannot invent headroom.

This notebook still includes Track A because David may want to audit whether Track A is truly closed, test a new architecture, or reproduce the decision contract.

### Track B: downstream dispatch control

Track B is the current positive lane. It is an operational extension: it exposes downstream dispatch controls that Garrido left fixed.

The key idea:

- The real bottleneck is downstream recovery / dispatch, especially around `Op10` and `Op12`.
- If the agent can control those dispatch multipliers, it can reduce backlog and recovery tail risk.
- The confirmed Track B result uses `track_b_v1`, observation `v7`, reward `control_v1`, risk level `adaptive_benchmark_v2`, horizon `h104`, 5 seeds, and dense static CRN comparison.

Best current paper-facing Track B baseline:

- action contract: `track_b_v1` with 8 continuous action dimensions;
- observation: `v7`;
- reward: `control_v1`;
- risk level: `adaptive_benchmark_v2`;
- horizon: `104` weekly decisions;
- primary metric: Garrido/Excel order-level ReT;
- secondary metrics: CVaR/tail, CTj/RPj/DPj, service loss, backlog, flow fill, Cobb-Douglas, and cost.

Do not call Track B thesis-faithful. It is a faithful extension of the DES bottleneck, not the original thesis decision surface.

## How to run this notebook

Recommended workflow:

1. Run setup.
2. Run the quick smoke tests for Track A and Track B.
3. Run the notebook-native editable model sandbox with tiny timesteps.
4. If it works, increase `TRAIN_TIMESTEPS` and maybe `TRAIN_SEEDS`.
5. For serious artifacts, run the repository CLI cells.

The defaults are intentionally small:

- `RUN_PROFILE = "smoke"` uses 1 seed and very few timesteps.
- `RUN_PROFILE = "debug"` is still small but more informative.
- `RUN_PROFILE = "serious"` is more expensive and should usually be done on Kaggle/Colab with a watcher.

For collaboration, keep changes inside the `CONFIG` cell first. Only edit model code after the smoke tests pass.

In [ ]:
# ============================================================
# 0) Central configuration cell
# ============================================================
# Edit this cell first. Most experiments can be changed here.

from __future__ import annotations
from pathlib import Path
import os
import sys
import json
import platform
import subprocess
from datetime import datetime, timezone

# --- Repository setup ---
# Use the active research branch by default because it contains Track B v9,
# rich metrics, Kaggle watcher fixes, and the latest Track B runners.
GIT_URL = "https://github.com/Thom-320/scres-ia.git"
GIT_BRANCH = "codex/garrido-replication-experiments"

# If this notebook is already inside the repo, set REPO_SOURCE="local".
# In Colab or Kaggle, use REPO_SOURCE="git".
REPO_SOURCE = "git"  # "git" | "local"
LOCAL_REPO_PATH = "/content/scres-ia"  # Colab default; Kaggle will switch to /kaggle/working/scres-ia below.

# --- Run profile ---
# smoke: fastest possible plumbing check.
# debug: still cheap, but enough to see non-random behavior.
# serious: not confirmatory by itself, but useful for a real cloud run.
RUN_PROFILE = "smoke"  # "smoke" | "debug" | "serious"

if RUN_PROFILE == "smoke":
    TRAIN_SEEDS = [1]
    TRAIN_TIMESTEPS = 512
    EVAL_EPISODES = 1
    MAX_STEPS = 8
    N_ENVS = 1
elif RUN_PROFILE == "debug":
    TRAIN_SEEDS = [1]
    TRAIN_TIMESTEPS = 5_000
    EVAL_EPISODES = 2
    MAX_STEPS = 26
    N_ENVS = 2
elif RUN_PROFILE == "serious":
    TRAIN_SEEDS = [1, 2]
    TRAIN_TIMESTEPS = 40_000
    EVAL_EPISODES = 4
    MAX_STEPS = 104
    N_ENVS = 4
else:
    raise ValueError(f"Unknown RUN_PROFILE={RUN_PROFILE!r}")

# --- Track A defaults ---
# Track A is for Garrido-style buffer/shift decisions.
# The most useful default is not blind PPO; it is a static headroom gate and/or
# a repair run with behavior-cloning from the best static frontier.
TRACK_A_DEFAULTS = {
    "lane": "boundary_characterization",
    "reward_mode": "ReT_excel_plus_cvar",
    "cvar_alpha": 0.1,
    "mode": "per_op",                  # per_op is the richer audit lane; continuous is common-buffer.
    "families": "R13,R24",             # useful conflict screen; change to R1,R2,R3,R24,mixed for broader gates.
    "phis": "1,2,4,8",
    "psis": "1.0",
    "fracs": "0,0.05,0.075,0.10,0.125,0.15,0.20,0.25",
    "shifts": "1,2,3",
    "max_steps": MAX_STEPS,
    "quick": True,
}

# Best-known Track A frozen reference from the per-op repair/conflict campaign.
# It is not a universal theorem; it is the current best audit reference.
TRACK_A_FROZEN_REFERENCE = {
    "description": "Best-known Track A static/per-op plateau from repair experiments; dynamic did not beat it robustly.",
    "action_family": "per_op_buffer + categorical shift",
    "op3_frac": 0.0,
    "op5_frac": 0.10,
    "op9_frac": 0.0,
    "shift": "S2",
    "best_static_excel_ret_approx": 0.155254,
    "best_dynamic_excel_ret_approx": 0.155247,
    "interpretation": "technical tie; no publishable Track A dynamic headroom so far",
}

# --- Track B defaults ---
# Paper-facing canonical baseline. This is the current positive lane.
TRACK_B_ENV_DEFAULTS = {
    "reward_mode": "control_v1",
    "risk_level": "adaptive_benchmark_v2",
    "observation_version": "v7",
    "action_contract": "track_b_v1",
    "step_size_hours": 168.0,
    "max_steps": MAX_STEPS,
}
TRACK_B_PPO_DEFAULTS = {
    "learning_rate": 1e-4,
    "n_steps": 256,
    "batch_size": 64,
    "n_epochs": 10,
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "clip_range": 0.2,
}
TRACK_B_DEFAULTS = {**TRACK_B_ENV_DEFAULTS, **TRACK_B_PPO_DEFAULTS}

# v8/v9 are candidates for headroom, not replacements for the canonical v7 claim.
TRACK_B_OBSERVATION_CANDIDATES = ["v7", "v8", "v9"]
TRACK_B_REWARD_CANDIDATES = ["control_v1", "ReT_excel_plus_cvar", "ReT_tail_v2", "ReT_garrido2024_train"]

# Best-known Track B frozen reference. This is a configuration/result reference,
# not a model checkpoint loader. It tells David what the current best result means.
TRACK_B_FROZEN_REFERENCE = {
    "description": "Canonical Track B confirmed result: PPO vs dense static CRN under adaptive_benchmark_v2.",
    "action_contract": "track_b_v1",
    "observation_version": "v7",
    "reward_mode": "control_v1",
    "risk_level": "adaptive_benchmark_v2",
    "horizon": "h104",
    "confirmed_train_seeds": 5,
    "confirmed_timesteps": 60_000,
    "order_ret_excel_ppo_approx": 0.00589,
    "order_ret_excel_best_static_approx": 0.00547,
    "delta_approx": 0.00043,
    "mechanism": "adaptive recovery / backlog control, not yet proven anticipation",
}

# --- Editable model defaults ---
# David can change MODEL_KIND to test custom models.
# "ppo_mlp" is the reliable baseline.
# "ppo_dmlpa" uses the DMLPA feature extractor cell below.
MODEL_KIND = "ppo_mlp"  # "ppo_mlp" | "ppo_dmlpa"
DMLPA_FACTOR = 1         # Use 1 for no frame stack. If using VecFrameStack, set to the number of stacked frames.
DMLPA_FEATURES_DIM = 120
DMLPA_NHEAD = 12
DMLPA_NUM_LAYERS = 4
DMLPA_HIDDEN = 100

# --- Outputs ---
RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_ROOT = Path("outputs/david_playground") / f"{RUN_PROFILE}_{RUN_TAG}"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("CONFIG loaded")
print(json.dumps({
    "RUN_PROFILE": RUN_PROFILE,
    "TRAIN_SEEDS": TRAIN_SEEDS,
    "TRAIN_TIMESTEPS": TRAIN_TIMESTEPS,
    "EVAL_EPISODES": EVAL_EPISODES,
    "MAX_STEPS": MAX_STEPS,
    "MODEL_KIND": MODEL_KIND,
    "OUTPUT_ROOT": str(OUTPUT_ROOT),
}, indent=2))

In [ ]:
# ============================================================
# 1) Portable setup for Colab, Kaggle, or local execution
# ============================================================
# This cell tries to be conservative:
# - If running in Colab/Kaggle, clone the repo.
# - If running locally inside the repo, reuse the current checkout.
# - Install requirements only when needed.

from pathlib import Path
import os
import sys
import subprocess
import platform


def detect_platform() -> str:
    if "COLAB_RELEASE_TAG" in os.environ:
        return "colab"
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle/working").exists():
        return "kaggle"
    return "local"

PLATFORM = detect_platform()
print("Detected platform:", PLATFORM)

if PLATFORM == "kaggle" and LOCAL_REPO_PATH == "/content/scres-ia":
    LOCAL_REPO_PATH = "/kaggle/working/scres-ia"

if REPO_SOURCE == "local":
    ROOT = Path.cwd()
else:
    ROOT = Path(LOCAL_REPO_PATH)
    if not ROOT.exists():
        print(f"Cloning {GIT_URL} branch {GIT_BRANCH} into {ROOT} ...")
        subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, GIT_URL, str(ROOT)], check=True)
    else:
        print(f"Repo already exists at {ROOT}; fetching branch {GIT_BRANCH} ...")
        subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", GIT_BRANCH, "--depth", "1"], check=False)
        subprocess.run(["git", "-C", str(ROOT), "checkout", GIT_BRANCH], check=False)

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# OUTPUT_ROOT is defined before setup so users can inspect it early.
# Re-create it after chdir so relative outputs live inside the repo checkout.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Working directory:", Path.cwd())
print("Python:", sys.version)
print("Branch:")
subprocess.run(["git", "branch", "--show-current"], check=False)

# Install dependencies. If everything is already installed, this is quick.
# If Colab/Kaggle has a conflict, restart runtime after install and rerun from the top.
INSTALL_DEPS = True
if INSTALL_DEPS:
    req = ROOT / "requirements.txt"
    if req.exists():
        print("Installing requirements.txt. This may take a few minutes on a fresh Colab/Kaggle runtime.")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=False)
    else:
        print("requirements.txt not found; installing minimal dependencies.")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "stable-baselines3", "sb3-contrib", "gymnasium", "simpy", "pandas", "numpy", "torch", "einops"], check=False)

print("Setup complete")

## Decision variables explained

### Track A variables

Track A asks whether RL can improve using Garrido-style decisions.

There are several Track A contracts in this repo:

1. **Thesis faithful DKANA 18D / 19D contract**
   - Action vector has 18 entries.
   - First 15 entries score inventory-period choices:
     - Op3: indices `0..4` for inventory periods `I168, I336, I504, I672, I1344`.
     - Op5: indices `5..9`.
     - Op9: indices `10..14`.
   - Last 3 entries score shifts:
     - `S1`: index `15`.
     - `S2`: index `16`.
     - `S3`: index `17`.
   - Default observation mode `decision_reward` is 19D: realized 18D decision vector plus reward.
   - This is the best handoff contract for David if the goal is thesis-faithful DKANA/DMLPA compatibility.

2. **Continuous Track A / per-op buffer contract**
   - Action may be `[common_buffer_frac, shift_signal]`, or richer `[op3_frac, op5_frac, op9_frac, shift_signal]`.
   - `shift_signal < -0.33` maps to `S1`, between `-0.33` and `0.33` maps to `S2`, and above `0.33` maps to `S3`.
   - This is useful for testing if PPO can find a narrow static plateau.

Recommended Track A audit defaults:

- Do not start with blind PPO.
- First run a static headroom gate.
- If there is no oracle-vs-constant headroom, do not train PPO.
- If there is headroom, use behavior cloning from the best static / oracle and then PPO fine-tuning.

### Track B variables

Track B uses `track_b_v1`, an 8D continuous action in `[-1, 1]`:

| Index | Variable | Meaning |
|---:|---|---|
| 0 | Op3 Q signal | upstream inventory/dispatch multiplier |
| 1 | Op9 Q signal | downstream supply-base quantity multiplier |
| 2 | Op3 ROP signal | reorder point extension |
| 3 | Op9 ROP signal | reorder point extension |
| 4 | Op5 Q signal | assembly input quantity multiplier |
| 5 | shift signal | maps to S1/S2/S3 via thresholds |
| 6 | Op10 dispatch signal | downstream dispatch multiplier, key Track B lever |
| 7 | Op12 dispatch signal | last-mile dispatch multiplier, key Track B lever |

The important Track B levers are indices `6` and `7` because they touch downstream dispatch and recovery. That is where the confirmed win lives.

In [ ]:
# ============================================================
# 2) Imports used by both Track A and Track B cells
# ============================================================
import json
import math
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from einops import rearrange

from stable_baselines3 import PPO
try:
    from stable_baselines3 import SAC
except Exception:
    SAC = None
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, VecFrameStack
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

from supply_chain.external_env_interface import (
    get_track_b_env_spec,
    get_dkana_thesis_faithful_env_spec,
    make_track_b_env,
    make_dkana_thesis_faithful_env,
    get_episode_terminal_metrics,
    get_observation_fields,
)
from supply_chain.continuous_its_env import (
    make_per_op_buffer_track_a_env,
    make_per_op_buffer_multidiscrete_track_a_env,
)
from supply_chain.episode_metrics import compute_episode_metrics

print("Imports OK")

In [ ]:
# ============================================================
# 3) Quick tests: environment contracts and one-step execution
# ============================================================
# These are intentionally short. They prove that the repo, imports, action spaces,
# and observation spaces are wired correctly before we spend time training.


def smoke_track_a_thesis_faithful() -> dict[str, Any]:
    env = make_dkana_thesis_faithful_env(
        reward_mode="ReT_seq_v1",
        risk_level="increased",
        observation_mode="decision_reward",
        max_steps=2,
        stochastic_pt=True,
    )
    obs, info = env.reset(seed=123)
    action = np.zeros(env.action_space.shape, dtype=np.float32)
    # A simple thesis-style action: choose one inventory period for Op3/Op5/Op9 and S2.
    # This is not claimed optimal; it is just a contract smoke test.
    action[2] = 1.0   # Op3 I504,1 score
    action[7] = 1.0   # Op5 I504,1 score
    action[12] = 1.0  # Op9 I504,1 score
    action[16] = 1.0  # S2 score
    next_obs, reward, terminated, truncated, info = env.step(action)
    result = {
        "action_shape": tuple(env.action_space.shape),
        "observation_shape": tuple(env.observation_space.shape),
        "reward": float(reward),
        "terminated": bool(terminated),
        "truncated": bool(truncated),
        "action_contract": getattr(env, "action_contract", "dkana_thesis_faithful"),
    }
    env.close()
    return result


def smoke_track_b() -> dict[str, Any]:
    env = make_track_b_env(**TRACK_B_ENV_DEFAULTS)
    obs, info = env.reset(seed=456)
    action = np.zeros(env.action_space.shape, dtype=np.float32)
    next_obs, reward, terminated, truncated, info = env.step(action)
    result = {
        "action_shape": tuple(env.action_space.shape),
        "observation_shape": tuple(env.observation_space.shape),
        "reward": float(reward),
        "terminated": bool(terminated),
        "truncated": bool(truncated),
        "observation_version": TRACK_B_DEFAULTS["observation_version"],
        "reward_mode": TRACK_B_DEFAULTS["reward_mode"],
        "risk_level": TRACK_B_DEFAULTS["risk_level"],
    }
    env.close()
    return result

track_a_smoke = smoke_track_a_thesis_faithful()
track_b_smoke = smoke_track_b()
print("Track A thesis-faithful smoke:")
print(json.dumps(track_a_smoke, indent=2))
print("\nTrack B smoke:")
print(json.dumps(track_b_smoke, indent=2))

assert track_a_smoke["action_shape"] == (18,), "Track A thesis-faithful action should be 18D"
assert track_b_smoke["action_shape"] == (8,), "Track B action should be 8D"
print("Smoke tests passed")

## Observation versions for Track B

Track B uses observation versions `v7`, `v8`, and `v9` most often.

- `v7` is the canonical confirmed baseline. It includes downstream bottleneck signals, rolling fill/backorder, and hazard features for R22/R23/R24.
- `v8` adds realized risk-ID observability: active/recent/duration features for R11-R14, R21-R24, and R3.
- `v9` adds queue health and trend features, including backorder queue count, unattended orders, oldest backorder age, EWMA fill/backlog, deltas, and previous produced/delivered/available hours.

Use `v7` when reproducing the current confirmed claim. Use `v8` or `v9` when searching for more headroom.

In [ ]:
# Inspect observation fields. This helps David know exactly what the model sees.
for obs_version in ["v7", "v8", "v9"]:
    fields = get_observation_fields(obs_version)
    print(f"\n{obs_version}: {len(fields)} fields")
    print(fields[:12], "...")
    print(fields[-12:])

## Editable DMLPA feature extractor

David likes to edit the model itself. This cell defines a DMLPA-style Transformer feature extractor compatible with Stable-Baselines3 policy kwargs.

Important notes:

- The observation dimension must be divisible by `factor`.
- If `factor=1`, DMLPA behaves like a one-token Transformer. It is mostly a compatibility smoke test.
- If you want real temporal context, wrap the vectorized environment with `VecFrameStack(..., n_stack=factor)` and set `DMLPA_FACTOR=factor`.
- The number of attention heads must divide `features_dim`.
- The default `features_dim=120` and `nhead=12` are compatible.

This class is intentionally close to the code David provided, with extra guards and comments.

In [ ]:
class DMLPA(BaseFeaturesExtractor):
    """DMLPA-style Transformer feature extractor for flattened observations.

    Expected input shape: (batch, factor * obs_dimension).
    It reshapes to (batch, factor, obs_dimension), embeds each token, adds
    sinusoidal positional encoding, runs a Transformer encoder, and pools the
    last token.
    """

    def __init__(
        self,
        observation_space,
        factor: int = 1,
        features_dim: int = 120,
        hidden_dim: int = 100,
        nhead: int = 12,
        num_layers: int = 4,
    ):
        super().__init__(observation_space, features_dim)
        if factor < 1:
            raise ValueError("factor must be >= 1")
        flat_dim = int(observation_space.shape[0])
        if flat_dim % factor != 0:
            raise ValueError(
                f"Observation dimension {flat_dim} is not divisible by factor={factor}. "
                "Use factor=1 or add VecFrameStack with a compatible stack count."
            )
        if features_dim % nhead != 0:
            raise ValueError("features_dim must be divisible by nhead")

        self.obs_dimension = flat_dim // factor
        self.factor = factor
        self.features_dim = features_dim

        self.latent_rw = torch.nn.Sequential(
            torch.nn.Linear(self.obs_dimension, hidden_dim),
            torch.nn.GELU(),
            torch.nn.Linear(hidden_dim, features_dim),
        )
        self.pre_norm = torch.nn.LayerNorm(features_dim)
        encoder_layer = torch.nn.TransformerEncoderLayer(
            d_model=features_dim,
            nhead=nhead,
            batch_first=True,
        )
        self.accumulated = torch.nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
        )
        self.register_buffer("pos_encoding", self.build_sinusoidal_pe(factor, features_dim))

    @staticmethod
    def build_sinusoidal_pe(seq_len: int, d_model: int) -> torch.Tensor:
        pe = torch.zeros(seq_len, d_model)
        position = torch.arange(0, seq_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)  # (1, seq_len, d_model)

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        observations = rearrange(observations, "b (d k) -> b d k", d=self.factor)
        observations = self.latent_rw(observations)
        observations = observations + self.pos_encoding
        observations = self.pre_norm(observations)
        observations = self.accumulated(observations)
        return observations[:, -1, :]


def build_policy_kwargs(model_kind: str) -> dict[str, Any] | None:
    if model_kind == "ppo_mlp":
        return None
    if model_kind == "ppo_dmlpa":
        return {
            "features_extractor_class": DMLPA,
            "features_extractor_kwargs": {
                "factor": DMLPA_FACTOR,
                "features_dim": DMLPA_FEATURES_DIM,
                "hidden_dim": DMLPA_HIDDEN,
                "nhead": DMLPA_NHEAD,
                "num_layers": DMLPA_NUM_LAYERS,
            },
            "net_arch": dict(pi=[128, 64], vf=[128, 64]),
        }
    raise ValueError(f"Unknown MODEL_KIND={model_kind}")

print("DMLPA class ready. MODEL_KIND =", MODEL_KIND)

In [ ]:
# ============================================================
# 4) Quick architecture shape test
# ============================================================
# This catches most DMLPA mistakes before training starts.

env_for_shape = make_track_b_env(**TRACK_B_ENV_DEFAULTS)
policy_kwargs = build_policy_kwargs(MODEL_KIND)
if MODEL_KIND == "ppo_dmlpa":
    extractor = DMLPA(
        env_for_shape.observation_space,
        factor=DMLPA_FACTOR,
        features_dim=DMLPA_FEATURES_DIM,
        hidden_dim=DMLPA_HIDDEN,
        nhead=DMLPA_NHEAD,
        num_layers=DMLPA_NUM_LAYERS,
    )
    sample = torch.as_tensor(env_for_shape.observation_space.sample()[None, :], dtype=torch.float32)
    out = extractor(sample)
    print("DMLPA output shape:", tuple(out.shape))
else:
    print("MODEL_KIND is ppo_mlp; no custom extractor shape test needed.")
env_for_shape.close()

## Notebook-native Track B training sandbox

This is the most editable part of the notebook.

Use this when David wants to change:

- PPO vs DMLPA feature extractor;
- learning rate;
- horizon;
- reward mode;
- observation version;
- number of seeds;
- timesteps;
- network size.

This sandbox is intentionally lightweight. It is not the final paper evaluator. It tells us whether a model can train and whether rewards/metrics move in a plausible direction.

For paper-grade evidence, use `scripts/run_track_b_smoke.py` or the dense CRN audit scripts.

In [ ]:
# ============================================================
# 5) Train a small Track B model inside the notebook
# ============================================================
# This is deliberately simple and easy to edit.


def make_track_b_training_env(seed: int):
    def _factory():
        env = make_track_b_env(**TRACK_B_ENV_DEFAULTS)
        env.reset(seed=seed)
        return Monitor(env)
    return _factory


def train_one_track_b_seed(seed: int):
    env_fns = [make_track_b_training_env(seed * 10_000 + i) for i in range(N_ENVS)]
    vec_env = DummyVecEnv(env_fns)

    # VecNormalize often helps PPO. For very tiny smoke runs, it is optional.
    use_vecnormalize = True
    if use_vecnormalize:
        vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.0)

    # Optional temporal stack for DMLPA. Keep disabled by default to avoid shape surprises.
    if MODEL_KIND == "ppo_dmlpa" and DMLPA_FACTOR > 1:
        vec_env = VecFrameStack(vec_env, n_stack=DMLPA_FACTOR)

    policy_kwargs = build_policy_kwargs(MODEL_KIND)
    model = PPO(
        "MlpPolicy",
        vec_env,
        learning_rate=TRACK_B_PPO_DEFAULTS["learning_rate"],
        n_steps=min(TRACK_B_PPO_DEFAULTS["n_steps"], 64 if RUN_PROFILE == "smoke" else TRACK_B_PPO_DEFAULTS["n_steps"]),
        batch_size=min(TRACK_B_PPO_DEFAULTS["batch_size"], 64),
        n_epochs=3 if RUN_PROFILE == "smoke" else TRACK_B_PPO_DEFAULTS["n_epochs"],
        gamma=TRACK_B_PPO_DEFAULTS["gamma"],
        gae_lambda=TRACK_B_PPO_DEFAULTS["gae_lambda"],
        clip_range=TRACK_B_PPO_DEFAULTS["clip_range"],
        policy_kwargs=policy_kwargs,
        verbose=1,
        seed=seed,
    )
    model.learn(total_timesteps=TRAIN_TIMESTEPS)
    return model, vec_env

trained_models = []
for seed in TRAIN_SEEDS:
    print(f"\nTraining Track B seed={seed}, MODEL_KIND={MODEL_KIND}, timesteps={TRAIN_TIMESTEPS}")
    model, vec_env = train_one_track_b_seed(seed)
    trained_models.append((seed, model, vec_env))

print("Training complete. Models in memory:", [seed for seed, _, _ in trained_models])

In [ ]:
# ============================================================
# 6) Deterministic evaluation for notebook-native models
# ============================================================
# This is a compact evaluator. It reports a small set of metrics and writes a CSV.
# For a full Garrido-style workbook, use the repository runner cells below.


def evaluate_track_b_model(model, seed: int, episodes: int = EVAL_EPISODES) -> list[dict[str, Any]]:
    rows = []
    for ep in range(episodes):
        eval_seed = 90_000 + seed * 1_000 + ep
        env = make_track_b_env(**TRACK_B_ENV_DEFAULTS)
        obs, info = env.reset(seed=eval_seed)
        done = False
        total_reward = 0.0
        steps = 0
        last_info = info
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += float(reward)
            done = bool(terminated or truncated)
            steps += 1
            last_info = info
        try:
            terminal = get_episode_terminal_metrics(env)
        except Exception as exc:
            terminal = {"terminal_metric_error": str(exc)}
        try:
            rich = compute_episode_metrics(env.unwrapped.sim)
        except Exception as exc:
            rich = {"rich_metric_error": str(exc)}
        row = {
            "seed": seed,
            "episode": ep,
            "eval_seed": eval_seed,
            "steps": steps,
            "reward_total": total_reward,
            "terminal_fill_rate": float(terminal.get("fill_rate", np.nan)),
            "terminal_backorder_rate": float(terminal.get("backorder_rate", np.nan)),
            "order_ret_excel": float(rich.get("order_level_ret_mean", np.nan)),
            "flow_fill_rate": float(rich.get("flow_fill_rate", np.nan)),
            "service_loss_auc_per_order": float(rich.get("service_loss_auc_per_order", np.nan)),
            "ctj_p99": float(rich.get("ctj_p99", np.nan)),
            "rpj_p99": float(rich.get("rpj_p99", np.nan)),
            "dpj_p99": float(rich.get("dpj_p99", np.nan)),
            "last_shifts_active": float(last_info.get("shifts_active", np.nan)),
        }
        rows.append(row)
        env.close()
    return rows

all_eval_rows = []
for seed, model, vec_env in trained_models:
    rows = evaluate_track_b_model(model, seed=seed, episodes=EVAL_EPISODES)
    all_eval_rows.extend(rows)

eval_df = pd.DataFrame(all_eval_rows)
display(eval_df)

out_csv = OUTPUT_ROOT / "notebook_track_b_eval.csv"
eval_df.to_csv(out_csv, index=False)
print("Wrote", out_csv)

## Repository runner path: Track B canonical smoke/audit

The notebook-native sandbox is good for editing the model. The repository runner is better when you need reproducible artifacts:

- `summary.json`
- `summary.md`
- `episode_metrics.csv`
- `seed_metrics.csv`
- `policy_summary.csv`
- optional per-order ledger

The command below uses the current Track B defaults. For a tiny run, keep `RUN_PROFILE="smoke"`. For a more serious run, change the central config to `debug` or `serious`.

Important:

- The primary metric is Excel/order-level ReT.
- Do not judge Track B only by `fill_rate`; it can be misleading.
- Always compare against static policies under the same eval seeds.

In [ ]:
# ============================================================
# 7) Launch the official Track B runner from the notebook
# ============================================================
# Set RUN_OFFICIAL_TRACK_B = True when you want to execute this cell.

RUN_OFFICIAL_TRACK_B = False

if RUN_OFFICIAL_TRACK_B:
    runner_out = OUTPUT_ROOT / "official_track_b_smoke"
    cmd = [
        sys.executable, "scripts/run_track_b_smoke.py",
        "--output-dir", str(runner_out),
        "--seeds", *[str(s) for s in TRAIN_SEEDS],
        "--train-timesteps", str(TRAIN_TIMESTEPS),
        "--eval-episodes", str(EVAL_EPISODES),
        "--reward-mode", TRACK_B_DEFAULTS["reward_mode"],
        "--risk-level", TRACK_B_DEFAULTS["risk_level"],
        "--observation-version", TRACK_B_DEFAULTS["observation_version"],
        "--max-steps", str(MAX_STEPS),
        "--n-envs", str(N_ENVS),
        "--learning-rate", str(TRACK_B_PPO_DEFAULTS["learning_rate"]),
        "--n-steps", str(TRACK_B_PPO_DEFAULTS["n_steps"]),
        "--batch-size", str(TRACK_B_PPO_DEFAULTS["batch_size"]),
    ]
    if RUN_PROFILE != "smoke":
        cmd.append("--export-order-ledger")
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    print("Runner output:", runner_out)
else:
    print("Skipped official Track B runner. Set RUN_OFFICIAL_TRACK_B=True to run it.")

## Track A audit cells

Track A is not the current positive lane, but David may want to test it.

Use these cells to:

1. verify the thesis-faithful DKANA 18D/19D contract;
2. run a small static headroom gate;
3. optionally run the repair PPO if a gate directory exists.

Do not spend large compute on Track A unless a static headroom gate first shows that oracle-per-regime beats best-constant. If static does not show headroom, PPO should not be expected to win.

In [ ]:
# ============================================================
# 8) Track A thesis-faithful DKANA smoke command
# ============================================================
# This is the cleanest David-facing Track A compatibility check.

RUN_TRACK_A_DKANA_SMOKE = True

if RUN_TRACK_A_DKANA_SMOKE:
    cmd = [
        sys.executable, "scripts/david_dkana_thesis_faithful_smoke.py",
        "--observation-mode", "decision_reward",
        "--reward-mode", "ReT_seq_v1",
        "--risk-level", "increased",
        "--max-steps", "2",
        "--seed", "42",
        "--stochastic-pt",
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("Skipped Track A DKANA smoke.")

In [ ]:
# ============================================================
# 9) Track A static headroom gate
# ============================================================
# This is the correct first question for Track A:
# Does a per-regime oracle beat the best single static action?
# If no, do not launch blind PPO.

RUN_TRACK_A_HEADROOM_GATE = False

if RUN_TRACK_A_HEADROOM_GATE:
    gate_out = OUTPUT_ROOT / "track_a_headroom_gate"
    cmd = [
        sys.executable, "scripts/run_track_a_headroom_search.py",
        "--mode", TRACK_A_DEFAULTS["mode"],
        "--families", TRACK_A_DEFAULTS["families"],
        "--phis", TRACK_A_DEFAULTS["phis"],
        "--psis", TRACK_A_DEFAULTS["psis"],
        "--fracs", TRACK_A_DEFAULTS["fracs"],
        "--shifts", TRACK_A_DEFAULTS["shifts"],
        "--seeds", ",".join(str(s) for s in TRAIN_SEEDS),
        "--max-steps", str(MAX_STEPS),
        "--output", str(gate_out),
    ]
    if TRACK_A_DEFAULTS["quick"]:
        cmd.append("--quick")
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    print("Gate output:", gate_out)
else:
    print("Skipped Track A headroom gate. Set RUN_TRACK_A_HEADROOM_GATE=True to run it.")

In [ ]:
# ============================================================
# 10) Optional Track A repair PPO, only after a gate exists
# ============================================================
# This reproduces the most sensible Track A repair logic:
# best-static teacher + behavior cloning + PPO fine-tune + checkpoint selection.
# It is disabled by default because Track A is not the current positive lane.

RUN_TRACK_A_REPAIR = False
TRACK_A_GATE_DIR = OUTPUT_ROOT / "track_a_headroom_gate"

if RUN_TRACK_A_REPAIR:
    if not TRACK_A_GATE_DIR.exists():
        raise FileNotFoundError(f"Gate dir does not exist: {TRACK_A_GATE_DIR}. Run the headroom gate first.")
    repair_out = OUTPUT_ROOT / "track_a_repair"
    cmd = [
        sys.executable, "scripts/run_track_a_training_repair.py",
        "--gate-dir", str(TRACK_A_GATE_DIR),
        "--reward-mode", TRACK_A_DEFAULTS["reward_mode"],
        "--cvar-alpha", str(TRACK_A_DEFAULTS["cvar_alpha"]),
        "--seeds", ",".join(str(s) for s in TRAIN_SEEDS),
        "--n-envs", str(N_ENVS),
        "--timesteps", str(TRAIN_TIMESTEPS),
        "--bc-epochs", "20" if RUN_PROFILE == "smoke" else "150",
        "--action-mode", "multidiscrete",
        "--teacher", "best_static",
        "--max-steps", str(MAX_STEPS),
        "--learning-rate", "0.0001",
        "--output", str(repair_out),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    print("Repair output:", repair_out)
else:
    print("Skipped Track A repair. Set RUN_TRACK_A_REPAIR=True after a gate passes.")

## Reward and observation sweep guidance

For fast iteration, do not try every possible combination. The practical sweep we have been using is:

- Rewards:
  - `control_v1`: current canonical Track B reward; simple and surprisingly strong.
  - `ReT_excel_plus_cvar`: directly tied to Excel ReT plus tail/service-loss penalty.
  - `ReT_tail_v2`: tail/recovery-focused.
  - `ReT_garrido2024_train`: Cobb-Douglas same-bar training candidate.
- Observations:
  - `v7`: confirmed baseline.
  - `v8`: adds realized risk IDs.
  - `v9`: adds backorder queue health and trends.
- Seeds:
  - smoke/debug: 1 or 2 seeds.
  - serious screen: 2 seeds.
  - confirmatory: 5+ seeds.

Promotion rule for Track B screens:

- promote only if Excel/order ReT beats the current baseline or if CVaR/tail improves strongly;
- always report cost separately;
- do not hide failures;
- do not claim retained learning unless H4 retained-vs-reset is actually positive.

In [ ]:
# ============================================================
# 11) Optional official Track B reward/observation sweep
# ============================================================
# This is heavier than the notebook-native sandbox. Use on Kaggle/Colab after smoke tests pass.

RUN_TRACK_B_SWEEP = False

if RUN_TRACK_B_SWEEP:
    sweep_out = OUTPUT_ROOT / "track_b_reward_obs_sweep"
    cmd = [
        sys.executable, "scripts/run_track_b_adaptive_sweep.py",
        "--reward-modes", ",".join(TRACK_B_REWARD_CANDIDATES),
        "--observation-versions", ",".join(TRACK_B_OBSERVATION_CANDIDATES),
        "--cvar-alphas", "0.05,0.1,0.2",
        "--seeds", *[str(s) for s in TRAIN_SEEDS],
        "--train-timesteps", str(TRAIN_TIMESTEPS),
        "--eval-episodes", str(EVAL_EPISODES),
        "--max-steps", str(MAX_STEPS),
        "--n-envs", str(N_ENVS),
        "--learning-rate", str(TRACK_B_PPO_DEFAULTS["learning_rate"]),
        "--output-dir", str(sweep_out),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    print("Sweep output:", sweep_out)
else:
    print("Skipped Track B sweep. Set RUN_TRACK_B_SWEEP=True to run it.")

## How we have been doing experiments

This project has been iterative and compute-aware.

Typical pattern:

1. **Smoke test**
   - 1 seed.
   - very short horizon or very few timesteps.
   - only checks that the env/action/model wiring works.

2. **Cheap screen**
   - 1-2 seeds.
   - 20k-40k timesteps.
   - 2-6 evaluation episodes.
   - used to decide whether a lane is alive.

3. **Repair/engineering screen**
   - behavior cloning / warm-start if the target is a narrow static plateau;
   - lower learning rate (`1e-4`);
   - multiple envs (`n_envs=4` when cloud resources allow);
   - checkpoint selection when possible.

4. **Confirmatory run**
   - 5+ seeds.
   - 60k-100k timesteps or more.
   - dense static frontier under common random numbers.
   - full Garrido-style metrics panel.

Track A tried many variants:

- static dense frontiers;
- continuous buffer fractions;
- per-op buffers;
- multidiscrete shift/buffer actions;
- behavior cloning from best static;
- reward variants including Excel+CVAR and tail rewards;
- risk-family and conflict campaigns.

The practical result was that Track A often reaches a plateau but does not robustly beat the best static frontier.

Track B became the positive lane because its actions reach the downstream bottleneck. The model reduces recovery tails and backlog instead of merely adding upstream buffer.

In [ ]:
# ============================================================
# 12) Save a run manifest for reproducibility
# ============================================================
manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "platform": PLATFORM,
    "git_url": GIT_URL,
    "git_branch": GIT_BRANCH,
    "run_profile": RUN_PROFILE,
    "train_seeds": TRAIN_SEEDS,
    "train_timesteps": TRAIN_TIMESTEPS,
    "eval_episodes": EVAL_EPISODES,
    "max_steps": MAX_STEPS,
    "n_envs": N_ENVS,
    "model_kind": MODEL_KIND,
    "dmlpa": {
        "factor": DMLPA_FACTOR,
        "features_dim": DMLPA_FEATURES_DIM,
        "nhead": DMLPA_NHEAD,
        "num_layers": DMLPA_NUM_LAYERS,
        "hidden": DMLPA_HIDDEN,
    },
    "track_a_defaults": TRACK_A_DEFAULTS,
    "track_a_frozen_reference": TRACK_A_FROZEN_REFERENCE,
    "track_b_defaults": TRACK_B_DEFAULTS,
    "track_b_frozen_reference": TRACK_B_FROZEN_REFERENCE,
}
manifest_path = OUTPUT_ROOT / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Wrote manifest:", manifest_path)
print(json.dumps(manifest, indent=2)[:2500])

## Interpretation checklist for David

After a run, ask these questions:

1. Did the environment smoke tests pass?
2. Did the action shape match the intended track?
   - Track A DKANA thesis faithful: 18D.
   - Track B: 8D.
3. Did training run without NaNs or action-shape errors?
4. Did the reward curve move in a plausible direction?
5. Did Excel/order ReT improve, or only the training reward?
6. Did service tail improve: CVaR, CTj/RPj/DPj p99, service-loss AUC?
7. Did the policy use the intended levers?
   - Track A: buffer/shift.
   - Track B: Op10/Op12 dispatch and shifts.
8. Did a dense static frontier beat the learned policy?
9. Is this a smoke result, a screen result, or a confirmatory result?

For paper language:

- Say Track A is a boundary characterization, not a failure of RL in general.
- Say Track B is an operational downstream-control extension.
- Say the current mechanism is adaptive recovery / backlog control.
- Do not claim anticipation or retained learning unless the lead/lag and H4 retained-vs-reset tests support it.